I recently read an article on sentence length in Greek hexameter poetry by [Dee Clayman from 1981](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1627358) ("Sentence Length in Greek Hexameter Poetry" in *Hexameter Studies*, *Quantitative Linguistics* 11)—an interesting article in many ways. It is a data-driven study and an excellent example of computational/philological/literary critical work in Classics from nearly four decades ago.

I could already do this sort of thing relatively easily for Latin hexameters using CLTK and the plaintext Latin Library corpus that I've [written about at *Disiecta Membra*](https://disiectamembra.wordpress.com/2016/08/11/working-with-the-latin-library-corpus-in-cltk/), but I didn't have a plaintext Greek corpus at hand. This notebook is the scraping infrastructure for assembling one—I'll leave the actual replication of Clayman's sentence-length analysis as a follow-up exercise.

I also just happened to teach a seminar last week on using Python to scrape XML by URL, and I thought this would make a good worked example.

The [Perseus Digital Library](http://www.perseus.tufts.edu) provides open-access XML texts of many of the hexameter texts from Clayman's article. We could, I suppose, cut and paste the texts from the browser. But for the *Odyssey* that would be almost three hundred pages chunked by section. Even chunked by book, we'd have to work through 24 pages.

Fortunately, the library also provides an XML Table of Contents which gives us a map of all the individual sections. With these TOC files, we can use Python to build a list of URLs for the sections, scrape these pages, extract lines of poetry, and finally stitch the results together. Python and [`lxml`](https://lxml.de) are well-suited to this task.

Below is a first pass at handling two Greek hexameter poems from Perseus: Homer's *Odyssey* and Hesiod's *Shield of Heracles*. The first is divided into 24 books; the second is a single self-contained text—the parser handles both shapes.

## Getting a plaintext *Odyssey*

In [ ]:
# Imports
import urllib.request
import time
from collections.abc import Iterable
from pprint import pprint

from lxml import etree

In [ ]:
# Constants
perseus_xml_base_url = 'http://www.perseus.tufts.edu/hopper/xmlchunk?doc='

# Homer's Odyssey TOC XML
odyssey_toc_url = 'http://www.perseus.tufts.edu/hopper/xmltoc?doc=Perseus%3Atext%3A1999.01.0135%3Abook%3D1%3Acard%3D1'

# Hesiod's Shield TOC XML
shield_toc_url = 'http://www.perseus.tufts.edu/hopper/xmltoc?doc=Perseus%3Atext%3A1999.01.0127%3Acard%3D1'

# The Hopper XML endpoints reject default urllib/curl user-agents; a
# browser-style UA is enough to get a 200.
UA = 'Mozilla/5.0 (compatible; ExploratoryPhilologyBlog/1.0; +https://exploratoryphilology.com)'


def fetch(url):
    req = urllib.request.Request(url, headers={'User-Agent': UA})
    with urllib.request.urlopen(req) as f:
        return f.read()

In [ ]:
def check_for_books(root):
    """Some poems are single self-contained works (e.g. Hesiod's Shield),
    others are divided into books (e.g. Homer's Odyssey). This tests for
    the presence of the attribute type='book' on a <chunk>, so book-level
    information can be retained when parsing."""
    return bool(root.findall(".//chunk[@type='book']"))

In [ ]:
perseus_toc_xml = fetch(odyssey_toc_url)
root = etree.fromstring(perseus_toc_xml)

In [ ]:
# Get list of refs from <chunk> elements

if check_for_books(root):
    books = root.findall(".//chunk[@type='book']")
    booknames = [book.find('head').text for book in books]
else:
    books = [root]
    booknames = ['work']

book_refs = []
for book in books:
    chunks = book.findall('chunk')
    refs = [chunk.attrib['ref'] for chunk in chunks]
    book_refs.append(refs)

In [ ]:
# Example refs retrieved from TOC for Odyssey 1
print(book_refs[0])

In [ ]:
# Get xml for each ref. Quarto's freeze: true will cache this so the
# fetch only happens once.
book_sections = []

for book_ref in book_refs:
    book_section_xml = []
    for ref in book_ref:
        time.sleep(0.1)
        book_section_xml.append(fetch(perseus_xml_base_url + ref))
    book_sections.append(book_section_xml)

In [ ]:
# Sample XML from Odyssey 1, Section 1
print(book_sections[0][0].decode('utf-8')[:1000])

In [ ]:
# Some helper functions

def check_for_lb(root):
    """Some poetry in the Perseus XML has lines delimited by <lb> and some
    by <l>. This tests for the presence of <lb>, so the right parser is
    used below."""
    return bool(root.findall('.//lb'))


def node_text(node):
    """Concatenate a node's own text with the tails of any inline children
    (e.g. intervening <milestone> elements inside an <l>).

    Adapted from https://stackoverflow.com/a/7500304
    """
    result = node.text or ''
    for child in node:
        if child.tail is not None:
            result += child.tail
    return result

In [ ]:
# Extract lines from each section
book_lines = []

for section in book_sections:
    section_lines = []
    for xml in section:
        root = etree.fromstring(xml)
        if check_for_lb(root):
            lines = root.findall('.//lb')
            lines = [line.tail for line in lines]
            lines = ['\n' if line is None else line for line in lines]
        else:
            lines = root.findall('.//l')
            lines = [node_text(line) for line in lines]
            lines = ['\n' if line is None else line for line in lines]
        section_lines.append(lines)
    book_lines.append(section_lines)

print(book_lines[0][0][:25])

In [ ]:
def flatten(seq):
    """Recursively flatten arbitrarily nested iterables of strings.

    Adapted from https://stackoverflow.com/a/2158532
    """
    for el in seq:
        if isinstance(el, Iterable) and not isinstance(el, (str, bytes)):
            yield from flatten(el)
        else:
            yield el

In [ ]:
plaintext = list(flatten(book_lines))
print('\n'.join(plaintext)[:1000])

## Getting a plaintext *Shield*

Same parser, different shape: Hesiod's *Shield* is a single self-contained text rather than a multi-book work, so `check_for_books` returns `False` and we treat the whole document as one 'book'.

In [ ]:
perseus_toc_xml = fetch(shield_toc_url)
root = etree.fromstring(perseus_toc_xml)

if check_for_books(root):
    books = root.findall(".//chunk[@type='book']")
    booknames = [book.find('head').text for book in books]
else:
    books = [root]
    booknames = ['work']

book_refs = []
for book in books:
    chunks = book.findall('chunk')
    refs = [chunk.attrib['ref'] for chunk in chunks]
    book_refs.append(refs)

In [ ]:
book_sections = []
for book_ref in book_refs:
    book_section_xml = []
    for ref in book_ref:
        time.sleep(0.1)
        book_section_xml.append(fetch(perseus_xml_base_url + ref))
    book_sections.append(book_section_xml)

In [ ]:
book_lines = []
for section in book_sections:
    section_lines = []
    for xml in section:
        root = etree.fromstring(xml)
        if check_for_lb(root):
            lines = root.findall('.//lb')
            lines = [line.tail for line in lines]
            lines = ['\n' if line is None else line for line in lines]
        else:
            lines = root.findall('.//l')
            lines = [node_text(line) for line in lines]
            lines = ['\n' if line is None else line for line in lines]
        section_lines.append(lines)
    book_lines.append(section_lines)

print(book_lines[0][0][:25])

In [ ]:
# Shield uses <lb>-delimited lines whose tails already include their own
# trailing newlines, so join with '' rather than '\n'.
plaintext = list(flatten(book_lines))
print(''.join(plaintext)[:1000])